# Prevent the Overdraft — Transactions Hands-On

**DBPRA · TU Berlin** · ~5 min

Two payments race against the same balance. The naive code lets it go below zero. **Your job: implement `withdraw_safe()` so it can't.**

In [ ]:
# Setup (~30s first run). Skip the details.
!sudo apt-get -qq install -y postgresql > /dev/null
!sudo service postgresql start
!sudo -u postgres psql -c "CREATE USER root SUPERUSER;" 2>/dev/null
!sudo -u postgres createdb demo 2>/dev/null
!pip install -q psycopg2-binary

import psycopg2, threading, time
from psycopg2 import errors as pgerr
DSN = "dbname=demo"

def reset_db():
    con = psycopg2.connect(DSN); con.autocommit = True
    cur = con.cursor()
    cur.execute("DROP TABLE IF EXISTS customer")
    cur.execute("CREATE TABLE customer (c_custkey INT PRIMARY KEY, c_acctbal NUMERIC)")
    cur.execute("INSERT INTO customer VALUES (17, 150)")
    con.close()

def balance():
    con = psycopg2.connect(DSN)
    cur = con.cursor()
    cur.execute("SELECT c_acctbal FROM customer WHERE c_custkey = 17")
    bal = cur.fetchone()[0]; con.close(); return bal

def run_concurrent(fn, n=2):
    results = []
    threads = [threading.Thread(target=lambda: results.append(fn())) for _ in range(n)]
    for t in threads: t.start()
    for t in threads: t.join()
    return results

reset_db()
print("Starting balance:", balance())

In [ ]:
# Naive: read → check → write, no transaction control. Watch the bug.
def withdraw_naive(amount=100):
    con = psycopg2.connect(DSN); con.autocommit = True
    cur = con.cursor()
    cur.execute("SELECT c_acctbal FROM customer WHERE c_custkey = 17")
    bal = cur.fetchone()[0]
    time.sleep(0.05)                                # widen the race
    if bal < amount:
        con.close(); return "rejected"
    cur.execute("UPDATE customer SET c_acctbal = c_acctbal - %s WHERE c_custkey = 17", (amount,))
    con.close(); return "ok"

reset_db()
print(run_concurrent(withdraw_naive))
print("Final balance:", balance())

### Your task

Fill in `withdraw_safe()` below so the balance can **never** go below 0.

Three things to write:

1. Set the **isolation level** to `SERIALIZABLE`
2. The transaction body — SELECT, check, UPDATE / commit, or rollback
3. The retry-handler cleanup when `SerializationFailure` fires

In [ ]:
def withdraw_safe(amount=100, max_retries=5):
    for attempt in range(max_retries):
        try:
            con = psycopg2.connect(DSN)
            # TODO 1: set isolation level to SERIALIZABLE
            cur = con.cursor()

            # TODO 2: SELECT balance (keep time.sleep(0.05) for fairness)
            #         if balance < amount → rollback + return "rejected"
            #         else                → UPDATE, commit, return "ok"

            con.close()
            return "TODO"
        except pgerr.SerializationFailure:
            # TODO 3: rollback the failed transaction, close the connection
            time.sleep(0.05 * (2 ** attempt))
    return "retries exhausted"

In [ ]:
reset_db()
print(run_concurrent(withdraw_safe))
print("Final balance:", balance())